# Augmented Images Viewer

View original sources and their augmented versions side by side.

## Import Libraries

In [9]:
import os
from pathlib import Path
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from IPython.display import display, clear_output
from collections import defaultdict

## Define Paths and Helper Functions

In [10]:
# Define paths
AUGMENTED_SOURCES_PATH = "/home/abigaildeklerk/Desktop/DeKlerk_Models/Data/AUGMENTED_SOURCES"
RADCAT_F_UNPACKED_PATH = "/home/abigaildeklerk/Desktop/DeKlerk_Models/Data/RADCAT_F_UNPACKED"

def load_image(image_path):
    """Load an image from file."""
    try:
        img = Image.open(image_path).convert('RGB')
        return np.array(img, dtype=np.uint8)
    except Exception as e:
        return None

def get_source_augmentations():
    """
    Organize augmented images by source ID.
    
    Returns:
        dict: {source_id: [list of augmentation paths]}
        list: sorted list of source_ids
    """
    source_augmentations = defaultdict(list)
    
    if not os.path.exists(AUGMENTED_SOURCES_PATH):
        return source_augmentations, []
    
    for file in sorted(os.listdir(AUGMENTED_SOURCES_PATH)):
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            # Extract source ID and augmentation index
            # Format: {source_id}_{aug_index}.png
            base_name = os.path.splitext(file)[0]
            
            # Find the last underscore followed by digits
            parts = base_name.rsplit('_', 1)
            if len(parts) == 2 and parts[1].isdigit():
                source_id = parts[0]
                file_path = os.path.join(AUGMENTED_SOURCES_PATH, file)
                source_augmentations[source_id].append(file_path)
    
    source_ids = sorted(source_augmentations.keys())
    return source_augmentations, source_ids

def get_original_image_path(source_id):
    """Get the path to the original RGB image."""
    original_path = os.path.join(RADCAT_F_UNPACKED_PATH, f"{source_id}_RGB.png")
    if os.path.exists(original_path):
        return original_path
    return None

## Display Functions

In [11]:
def display_source_with_augmentations(source_id, source_augmentations):
    """
    Display the original image and all augmented images for a source.
    
    Args:
        source_id (str): Source ID
        source_augmentations (dict): Dictionary mapping source_id to list of augmentation paths
    """
    try:
        augmentation_paths = source_augmentations.get(source_id, [])
        original_path = get_original_image_path(source_id)
        
        # Calculate grid layout: 1 original + augmentations
        # Use 6 columns, with original taking 1 spot, augmentations taking the rest
        num_images = 1 + len(augmentation_paths)  # original + augmentations
        num_cols = 6
        num_rows = (num_images + num_cols - 1) // num_cols  # Ceiling division
        
        fig = plt.figure(figsize=(20, 3.5 * num_rows))
        gs = gridspec.GridSpec(num_rows, num_cols, figure=fig, hspace=0.4, wspace=0.3)
        
        # Title
        fig.suptitle(f'Source {source_id}: Original + {len(augmentation_paths)} Augmentations', 
                     fontsize=16, fontweight='bold')
        
        # Display original image
        ax = fig.add_subplot(gs[0, 0])
        if original_path:
            original_img = load_image(original_path)
            if original_img is not None:
                ax.imshow(original_img)
                ax.set_title('Original', fontsize=11, fontweight='bold', color='green')
            else:
                ax.text(0.5, 0.5, 'Could not load\noriginal', ha='center', va='center',
                       transform=ax.transAxes, fontsize=10)
                ax.set_title('Original', fontsize=11)
        else:
            ax.text(0.5, 0.5, 'Original\nnot found', ha='center', va='center',
                   transform=ax.transAxes, fontsize=10)
            ax.set_title('Original', fontsize=11)
        
        ax.axis('off')
        
        # Display augmentation images
        for idx, aug_path in enumerate(augmentation_paths):
            # Calculate position in grid (skip first position which is original)
            grid_idx = idx + 1
            row = grid_idx // num_cols
            col = grid_idx % num_cols
            
            ax = fig.add_subplot(gs[row, col])
            
            aug_img = load_image(aug_path)
            if aug_img is not None:
                ax.imshow(aug_img)
                ax.set_title(f'Aug {idx}', fontsize=10)
            else:
                ax.text(0.5, 0.5, f'Error\nloading', ha='center', va='center',
                       transform=ax.transAxes, fontsize=9)
                ax.set_title(f'Aug {idx}', fontsize=10)
            
            ax.axis('off')
        
        plt.show()
        
    except Exception as e:
        print(f"Error displaying source {source_id}: {str(e)}")

## Interactive Navigation

In [12]:
# Load source augmentations data
source_augmentations, source_ids = get_source_augmentations()

if not source_ids:
    print("No augmented sources found in AUGMENTED_SOURCES folder!")
else:
    print(f"Found {len(source_ids)} sources with augmentations")
    
    # State variables
    current_index = 0
    
    # Create widgets
    source_dropdown = widgets.Dropdown(
        options=source_ids,
        value=source_ids[0],
        description='Source ID:',
        style={'description_width': '80px'}
    )
    
    prev_button = widgets.Button(
        description='← Previous',
        button_style='info',
        tooltip='Show previous source'
    )
    
    next_button = widgets.Button(
        description='Next →',
        button_style='info',
        tooltip='Show next source'
    )
    
    status_label = widgets.HTML(
        value=f'<b>Status:</b> {source_ids[0]} | {len(source_augmentations[source_ids[0]])} augmentations (1 of {len(source_ids)})'
    )
    
    display_output = widgets.Output()
    
    def on_prev_clicked(b):
        """Handle previous button click."""
        global current_index
        current_index = (current_index - 1) % len(source_ids)
        source_dropdown.value = source_ids[current_index]
    
    def on_next_clicked(b):
        """Handle next button click."""
        global current_index
        current_index = (current_index + 1) % len(source_ids)
        source_dropdown.value = source_ids[current_index]
    
    def on_dropdown_changed(change):
        """Handle source dropdown change."""
        global current_index
        current_index = source_ids.index(change['new'])
        update_display()
    
    def update_display():
        """Update the display with the selected source."""
        source_id = source_dropdown.value
        num_augs = len(source_augmentations[source_id])
        current_num = current_index + 1
        status_label.value = f'<b>Status:</b> {source_id} | {num_augs} augmentations ({current_num} of {len(source_ids)})'
        
        with display_output:
            clear_output(wait=True)
            display_source_with_augmentations(source_id, source_augmentations)
    
    # Attach event handlers
    prev_button.on_click(on_prev_clicked)
    next_button.on_click(on_next_clicked)
    source_dropdown.observe(on_dropdown_changed, names='value')
    
    # Create layout
    dropdown_box = widgets.HBox([source_dropdown], layout=widgets.Layout(justify_content='center', margin='10px'))
    nav_buttons = widgets.HBox([prev_button, next_button], layout=widgets.Layout(justify_content='center', margin='10px'))
    
    # Display the interface
    print("\n" + "=" * 80)
    print("AUGMENTED SOURCES VIEWER")
    print("=" * 80)
    display(dropdown_box)
    display(nav_buttons)
    display(status_label)
    display(display_output)
    
    # Show initial display
    update_display()

Found 2008 sources with augmentations

AUGMENTED SOURCES VIEWER


HTML(value='<b>Status:</b> 10000 | 4 augmentations (1 of 2008)')

Output()